# Cell Fate Dynamics: Exploring Pancreas Differentiation with scRNA-seq

This notebook provides a self-contained exploration of cell-state transitions and latent representations using the public scVelo pancreas dataset.

### Objectives:
1. **Explore and preprocess** spliced and unspliced single-cell counts.
2. **Learn and compare low-dimensional representations** (PCA baseline vs. scVI latent space).
3. **Estimate transition directions** using scVelo (RNA velocity) and terminal-state probabilities with CellRank.
4. **Evaluate robustness quantitatively** rather than relying on 2D embeddings alone.
5. **Discuss limitations** and uncertainties of velocity estimation.

## 0. Environment Setup (Google Colab / Local)

In [ ]:
# In Google Colab, uncomment and run:
# !pip install scanpy scvelo cellrank scvi-tools matplotlib seaborn session-info

import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import scvelo as scv
import scvi
import cellrank as cr
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors

# Set visual styles & reproducibility seeds
scv.settings.verbosity = 3
scv.settings.set_figure_params('scvelo', dpi=100)
sc.settings.verbosity = 2
RANDOM_SEED = 42
scvi.settings.seed = RANDOM_SEED
np.random.seed(RANDOM_SEED)

## 1. Data Exploration & Quality Control

We use the scVelo endocrine pancreas differentiation dataset (Bastidas-Ponce et al., 2019).
It contains lineage commitment of endocrine cells into alpha, beta, delta, and epsilon cells.

In [ ]:
# Preserve raw counts layer for scVI
adata.layers['counts'] = adata.X.copy()

from cell_fate_dynamics.data import preprocess_data
adata = preprocess_data(adata, min_shared_counts=20, n_top_genes=2000)
print('Filtered AnnData shape:', adata.shape)


In [ ]:
# Inspect spliced vs unspliced counts distribution
scv.pl.proportions(adata)

from cell_fate_dynamics.models import train_scvi
adata, vae = train_scvi(adata, layer='counts', n_latent=30, max_epochs=45, batch_size=512, seed=RANDOM_SEED, early_stopping=True, early_stopping_patience=8)
print('scVI latent shape:', adata.obsm['X_scVI'].shape)


In [ ]:
# Preserve raw counts layer for scVI
adata.layers['counts'] = adata.X.copy()

# Filter genes and normalize
scv.pp.filter_and_normalize(adata, min_shared_counts=20, n_top_genes=2000)
sc.tl.pca(adata, svd_solver='arpack', random_state=RANDOM_SEED)
sc.pp.neighbors(adata, n_pcs=30, n_neighbors=30, random_state=RANDOM_SEED)
scv.pp.moments(adata, n_pcs=30, n_neighbors=30)
print('Filtered AnnData shape:', adata.shape)

## 3. Representation Learning: PCA vs. scVI (Variational Autoencoder)

We compare standard linear PCA with the non-linear deep generative model scVI.

In [ ]:
# Train scVI model
scvi.model.SCVI.setup_anndata(adata, layer='counts')
vae = scvi.model.SCVI(adata, n_latent=30)
vae.train(max_epochs=200, early_stopping=True, plan_kwargs={'lr': 1e-3})

# Extract latent representation
adata.obsm['X_scVI'] = vae.get_latent_representation(adata)
print('scVI latent shape:', adata.obsm['X_scVI'].shape)

In [ ]:
from cell_fate_dynamics.models import compute_velocity
adata = compute_velocity(adata, mode='stochastic', basis='umap_pca', n_jobs=-1)
scv.tl.velocity_embedding(adata, basis='umap_scvi')

# Velocity streams on PCA-derived UMAP
scv.pl.velocity_embedding_stream(adata, basis='umap_pca', color='clusters', title='RNA Velocity Stream (PCA UMAP)')


### Quantitative Comparison: Silhouette & Neighborhood Overlap
We test how well cell types are separated and whether local geometry is preserved.

In [ ]:
from cell_fate_dynamics.evaluation import evaluate_representation_clustering, neighborhood_overlap

sil_scores = evaluate_representation_clustering(adata, cell_type_key='clusters', rep_keys=['X_pca', 'X_scVI'])
overlap = neighborhood_overlap(adata.obsm['X_pca'], adata.obsm['X_scVI'], n_neighbors=15)

print('Silhouette scores (higher indicates clearer cluster separation):')
for k, v in sil_scores.items():
    print(f'  {k}: {v:.4f}')
print(f'Mean k-NN Jaccard neighborhood overlap (k=15): {overlap:.4f}')

vk = cr.kernels.VelocityKernel(adata)
vk.compute_transition_matrix(n_jobs=-1, show_progress_bar=False)
ck = cr.kernels.ConnectivityKernel(adata)
ck.compute_transition_matrix()
combined_kernel = 0.8 * vk + 0.2 * ck

g = cr.estimators.GPCCA(combined_kernel)
g.compute_macrostates(n_states=5, cluster_key='clusters')
adata.obs['macrostates'] = g.macrostates
sc.pl.embedding(adata, basis='umap_pca', color='macrostates', title='CellRank Inferred Macrostates (GPCCA)')


In [ ]:
# Estimate velocity using stochastic model
scv.tl.velocity(adata, mode='stochastic')
scv.tl.velocity_graph(adata)

# Velocity streams on PCA-derived UMAP
scv.pl.velocity_embedding_stream(adata, basis='umap_pca', color='clusters', title='RNA Velocity Stream (PCA UMAP)')

In [ ]:
# Robustness test: velocity consistency under random gene subsampling (80%)
rng = np.random.default_rng(RANDOM_SEED)
subsample_idx = rng.choice(adata.n_vars, size=int(adata.n_vars * 0.8), replace=False)
adata_sub = adata[:, subsample_idx].copy()

scv.tl.velocity(adata_sub, mode='stochastic')
scv.tl.velocity_graph(adata_sub, n_jobs=-1)
scv.tl.velocity_confidence(adata_sub)

corr = np.corrcoef(adata.obs['velocity_confidence'].values, adata_sub.obs['velocity_confidence'].values)[0, 1]
print(f'Pearson correlation of velocity confidence between full and 80% gene subset: {corr:.4f}')


## 5. Terminal States and CellRank

CellRank combines RNA velocity with transcriptomic similarity to infer fate probabilities and initial/terminal states.

In [ ]:
vk = cr.kernels.VelocityKernel(adata)
ck = cr.kernels.ConnectivityKernel(adata)
combined_kernel = 0.8 * vk + 0.2 * ck

g = cr.estimators.GPCCA(combined_kernel)
g.compute_macrostates(n_states=5, cluster_key='clusters')
g.plot_macrostates(basis='umap_pca')

## 6. Robustness & Sensitivity Checks

We assess whether the inferred dynamics are stable under gene downsampling and random perturbations.

In [ ]:
# Robustness test: velocity consistency under random gene subsampling (80%)
rng = np.random.default_rng(RANDOM_SEED)
subsample_idx = rng.choice(adata.n_vars, size=int(adata.n_vars * 0.8), replace=False)
adata_sub = adata[:, subsample_idx].copy()

scv.tl.velocity(adata_sub, mode='stochastic')
scv.tl.velocity_graph(adata_sub)
scv.tl.velocity_confidence(adata_sub)

corr = np.corrcoef(adata.obs['velocity_confidence'].values, adata_sub.obs['velocity_confidence'].values)[0, 1]
print(f'Pearson correlation of velocity confidence between full and 80% gene subset: {corr:.4f}')

## 7. Limitations and Discussion

- **Velocity Assumptions**: RNA velocity assumes constant transcription, splicing, and degradation rates, which are often violated during transient cell state transitions.
- **Embedding Discrepancies**: 2D UMAP projection can visually exaggerate continuity or distort trajectory directions.
- **scVI vs. PCA**: scVI accounts for non-linear count distributions and library size variation, yielding tighter cluster boundaries, while PCA provides an interpretable linear projection.